In [1]:
import pandas as pd
import numpy as np
import torch.nn as nn
from torch_geometric.nn import SAGEConv, GATConv
import torch
from torch_geometric.data import HeteroData
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from collections import Counter
np.random.seed(42)

c:\Users\tusha\miniconda3\envs\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_DIR = "datasets"

In [3]:
students = pd.read_csv(f"{DATA_DIR}/students.csv")
courses = pd.read_csv(f"{DATA_DIR}/courses.csv")
course_prerequisites = pd.read_csv(f"{DATA_DIR}/course_prerequisites.csv")
concepts = pd.read_csv(f"{DATA_DIR}/concepts.csv")
concept_prerequisites = pd.read_csv(f"{DATA_DIR}/concept_prerequisites.csv")
course_concepts = pd.read_csv(f"{DATA_DIR}/course_concepts.csv")
enrollments = pd.read_csv(f"{DATA_DIR}/enrollments.csv")
assessment_scores = pd.read_csv(f"{DATA_DIR}/assessment_scores.csv")
chatbot_signals = pd.read_csv(f"{DATA_DIR}/chatbot_signals.csv")
graph_edges = pd.read_csv(f"{DATA_DIR}/graph_edges.csv")

In [4]:
for name, df in [("students", students), ("courses", courses), ("concepts", concepts), 
                  ("enrollments", enrollments), ("chatbot_signals", chatbot_signals)]:
    print(f"{name}: {df.shape}  |  cols: {list(df.columns)}")

students: (2000, 7)  |  cols: ['student_id', 'base_ability', 'effort_level', 'true_modality', 'gpa_start', 'risk_factor', 'split']
courses: (40, 7)  |  cols: ['course_id', 'course_name', 'tier', 'department', 'best_modality', 'difficulty', 'pass_rate']
concepts: (80, 4)  |  cols: ['concept_id', 'concept_name', 'domain', 'difficulty']
enrollments: (41985, 7)  |  cols: ['student_id', 'course_id', 'semester', 'grade', 'passed', 'modality_match', 'risk_class']
chatbot_signals: (35006, 7)  |  cols: ['student_id', 'concept_id', 'num_questions', 'confusion_score', 'help_requests', 'recency_days', 'help_style']


In [5]:
chatbot_agg = chatbot_signals.groupby('student_id').agg(
    avg_confusion = ('confusion_score', 'mean'),
    avg_questions = ('num_questions','mean')
).reset_index()

chatbot_agg = chatbot_agg.set_index('student_id').reindex(students['student_id']).reset_index()

print(chatbot_agg.head())
print(f"Shape: {chatbot_agg.shape}")

  student_id  avg_confusion  avg_questions
0      S0000       0.256250       4.392857
1      S0001       0.294917       5.916667
2      S0002       0.258690       4.413793
3      S0003       0.393217       6.739130
4      S0004       0.099889       2.944444
Shape: (2000, 3)


In [6]:
# def normalize(df, cols):
#     scaler = StandardScaler()
#     array = scaler.fit_transform(df[cols].values)
#     array_x = torch.tensor(array, dtype = torch.float)

#     print(array_x.shape)
#     return array_x

def normalize(df, cols, fit_mask = None):
    scaler = StandardScaler()
    values = df[cols].values
    fit_data = values[fit_mask] if fit_mask is not None else values
    scaler.fit(fit_data)
    array = scaler.transform(values)
    return torch.tensor(array, dtype = torch.float)

In [7]:
course_emb_array = np.load('datasets/course_embeddings.npy')
concept_emb_array = np.load('datasets/concept_embeddings.npy')

In [8]:
train_bool = (students['split'] == 'train').values
students_cols = ['gpa_start', 'effort_level','risk_factor']
students_x = normalize(students, students_cols, fit_mask=train_bool)

chatbot_cols = ['avg_confusion', 'avg_questions']
chatbot_x = normalize(chatbot_agg, chatbot_cols, fit_mask = train_bool)

students_x = torch.cat([students_x, chatbot_x], dim=1)
print("Student features shape:", students_x.shape)

courses_cols = ['tier', 'difficulty', 'pass_rate']
courses_numeric = normalize(courses, courses_cols)

modality_onehot = pd.get_dummies(courses['best_modality']).values
modality_tensor = torch.tensor(modality_onehot, dtype = torch.float)

print("Numeric shape:", courses_numeric.shape)
print("one-hot shape:", modality_tensor.shape)
course_emb_tensor = torch.tensor(course_emb_array, dtype=torch.float)

courses_x = torch.cat([courses_numeric, modality_tensor, course_emb_tensor], dim=1)
print("Course features:", courses_x.shape) 

concepts_cols = ['difficulty']
concepts_numeric = normalize(concepts, concepts_cols)

domain_onehot = pd.get_dummies(concepts['domain']).values
domain_tensor = torch.tensor(domain_onehot, dtype=torch.float)

print("Numeric shape:", concepts_numeric.shape) 
print("One-hot shape:", domain_tensor.shape) 

concept_emb_tensor = torch.tensor(concept_emb_array, dtype=torch.float)

concepts_x = torch.cat([concepts_numeric, domain_tensor, concept_emb_tensor], dim=1)
print("Concept features:", concepts_x.shape)

Student features shape: torch.Size([2000, 5])
Numeric shape: torch.Size([40, 3])
one-hot shape: torch.Size([40, 4])
Course features: torch.Size([40, 1031])
Numeric shape: torch.Size([80, 1])
One-hot shape: torch.Size([80, 6])
Concept features: torch.Size([80, 1031])


In [9]:
print(students_x)

tensor([[ 0.3031,  0.7477,  0.6678, -0.2802, -0.4871],
        [ 0.0850, -0.3068, -0.2730,  0.0316,  0.6082],
        [ 0.7591,  0.0952, -0.8317, -0.2605, -0.4721],
        ...,
        [ 0.6005,  1.0949, -0.8819, -0.5629,  0.1410],
        [-1.1442, -1.5716,  1.2960,  0.3040,  1.0929],
        [ 0.8979,  0.7913, -1.3556, -1.4446, -0.9809]])


In [10]:
student_id_map = {sid: idx for idx, sid in enumerate(students['student_id'])}
course_id_map = {cid: idx for idx, cid in enumerate(courses['course_id'])}
concept_id_map = {cid: idx for idx, cid in enumerate(concepts['concept_id'])}

In [11]:
print(graph_edges['edge_type'].unique())

['enrolled_in' 'engaged_with' 'prereq_of' 'covers' 'concept_prereq']


In [12]:
def making_index(src_map, dest_map, edges):
    src = [src_map[i] for i in edges['src_id']]
    dst = [dest_map[i] for i in edges['dst_id']]

    edge_index = torch.tensor([src, dst], dtype = torch.long)
    edge_weight = torch.tensor(edges['weight'].values, dtype=torch.float)

    print(edge_index.shape, edge_weight.shape)

    return edge_index, edge_weight

In [13]:
enrolled_edges = graph_edges[graph_edges['edge_type'] == 'enrolled_in']
engaged_edges = graph_edges[graph_edges['edge_type'] == 'engaged_with']
prereq_edges = graph_edges[graph_edges['edge_type'] == 'prereq_of']
covers_edges = graph_edges[graph_edges['edge_type'] == 'covers']
concept_prereq_edges = graph_edges[graph_edges['edge_type'] == 'concept_prereq']

In [14]:
enrolled_edge_index, enrolled_edge_weight = making_index(student_id_map, course_id_map, enrolled_edges)
engaged_edge_index, engaged_edge_weight = making_index(student_id_map, concept_id_map, engaged_edges)
prereq_edge_index, prereq_edge_weight = making_index(course_id_map, course_id_map, prereq_edges)
cover_edge_index, cover_edge_weight = making_index(course_id_map, concept_id_map, covers_edges)
concept_prereq_edge_index, concept_prereq_edge_weight = making_index(concept_id_map, concept_id_map, concept_prereq_edges)

torch.Size([2, 41985]) torch.Size([41985])
torch.Size([2, 35006]) torch.Size([35006])
torch.Size([2, 64]) torch.Size([64])
torch.Size([2, 157]) torch.Size([157])
torch.Size([2, 89]) torch.Size([89])


In [15]:
data = HeteroData()

data['student'].x = students_x
data['course'].x = courses_x
data['concept'].x = concepts_x

data['student', 'enrolled_in', 'course'].edge_index = enrolled_edge_index
data['student', 'enrolled_in', 'course'].edge_weight = enrolled_edge_weight
 
data['student', 'engaged_with', 'concept'].edge_index = engaged_edge_index
data['student', 'engaged_with', 'concept'].edge_weight = engaged_edge_weight
 
data['course', 'prereq_of', 'course'].edge_index = prereq_edge_index
data['course', 'prereq_of', 'course'].edge_weight = prereq_edge_weight
 
data['course', 'covers', 'concept'].edge_index = cover_edge_index
data['course', 'covers', 'concept'].edge_weight = cover_edge_weight
 
data['concept', 'concept_prereq', 'concept'].edge_index = concept_prereq_edge_index
data['concept', 'concept_prereq', 'concept'].edge_weight = concept_prereq_edge_weight

data['course', 'rev_enrolled_in', 'student'].edge_index = enrolled_edge_index.flip(0)
data['concept','rev_engaged_with', 'student'].edge_index = engaged_edge_index.flip(0)
data['course', 'rev_enrolled_in', 'student'].edge_weight = enrolled_edge_weight
data['concept', 'rev_engaged_with', 'student'].edge_weight = engaged_edge_weight
 

In [16]:
print(data)

HeteroData(
  student={ x=[2000, 5] },
  course={ x=[40, 1031] },
  concept={ x=[80, 1031] },
  (student, enrolled_in, course)={
    edge_index=[2, 41985],
    edge_weight=[41985],
  },
  (student, engaged_with, concept)={
    edge_index=[2, 35006],
    edge_weight=[35006],
  },
  (course, prereq_of, course)={
    edge_index=[2, 64],
    edge_weight=[64],
  },
  (course, covers, concept)={
    edge_index=[2, 157],
    edge_weight=[157],
  },
  (concept, concept_prereq, concept)={
    edge_index=[2, 89],
    edge_weight=[89],
  },
  (course, rev_enrolled_in, student)={
    edge_index=[2, 41985],
    edge_weight=[41985],
  },
  (concept, rev_engaged_with, student)={
    edge_index=[2, 35006],
    edge_weight=[35006],
  }
)


In [17]:
train_mask = torch.tensor(students['split'] == 'train', dtype = torch.bool)
val_mask = torch.tensor(students['split'] == 'val', dtype = torch.bool)
test_mask = torch.tensor(students['split'] == 'test', dtype = torch.bool)

data['student'].train_mask = train_mask
data['student'].val_mask = val_mask
data['student'].test_mask = test_mask

print(train_mask.sum())
print(val_mask.sum())
print(test_mask.sum())

tensor(1400)
tensor(300)
tensor(300)


In [18]:
## TSNE

class GraphSAGE(torch.nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()

        self.conv1 = nn.ModuleDict({
            'enrolled_in': GATConv((-1, -1), hidden_dim, edge_dim=1, add_self_loops=False),
            'engaged_with': GATConv((-1, -1), hidden_dim, edge_dim=1, add_self_loops=False),
            'prereq_of': SAGEConv(-1, hidden_dim),
            'covers': SAGEConv(-1, hidden_dim),
            'concept_prereq': SAGEConv(-1, hidden_dim),
            'rev_enrolled_in': GATConv((-1, -1), hidden_dim, edge_dim=1, add_self_loops=False),
            'rev_engaged_with': GATConv((-1, -1), hidden_dim, edge_dim=1, add_self_loops=False)
        })

        self.conv2 = nn.ModuleDict({
            'rev_enrolled_in': SAGEConv(hidden_dim, hidden_dim),
            'rev_engaged_with': SAGEConv(hidden_dim, hidden_dim)
        })

        self.dropout = nn.Dropout(0.3)
        self.student_proj = nn.Linear(5, hidden_dim)
        self.layer_norm = nn.LayerNorm(hidden_dim)

        self.task_a_head = nn.Linear(hidden_dim*4, 3)
        # self.task_a_head = nn.Sequential(
        #     nn.Linear(hidden_dim * 2, hidden_dim),
        #     nn.ReLU(),
        #     nn.Dropout(0.3), 
        #     nn.Linear(hidden_dim, 32),
        #     nn.ReLU(),
        #     nn.Dropout(0.3),
        #     nn.Linear(32, 3)
        # )

    def forward(self, data):

        x_student = data['student'].x
        x_course = data['course'].x
        x_concept = data['concept'].x

        ## Layer 1 
        course_from_students = self.conv1['enrolled_in'](
            (x_student, x_course),
            data['student', 'enrolled_in', 'course'].edge_index,
            edge_attr=data['student', 'enrolled_in', 'course'].edge_weight.unsqueeze(1)
        )
        course_from_course = self.conv1['prereq_of'](
            (x_course, x_course),
            data['course', 'prereq_of', 'course'].edge_index
        )
        concepts_from_students = self.conv1['engaged_with'](
            (x_student, x_concept),
            data['student', 'engaged_with', 'concept'].edge_index,
            edge_attr=data['student', 'engaged_with', 'concept'].edge_weight.unsqueeze(1)
        )
        concepts_from_course = self.conv1['covers'](
            (x_course, x_concept),
            data['course', 'covers', 'concept'].edge_index
        )
        concepts_from_concepts = self.conv1['concept_prereq'](
            (x_concept, x_concept),
            data['concept', 'concept_prereq', 'concept'].edge_index
        )
        student_from_course_l1 = self.conv1['rev_enrolled_in'](
            (x_course, x_student),
            data['course', 'rev_enrolled_in', 'student'].edge_index,
            edge_attr=data['course', 'rev_enrolled_in', 'student'].edge_weight.unsqueeze(1)
        )
        student_from_concept_l1 = self.conv1['rev_engaged_with'](
            (x_concept, x_student),
            data['concept', 'rev_engaged_with', 'student'].edge_index,
            edge_attr=data['concept', 'rev_engaged_with', 'student'].edge_weight.unsqueeze(1)
        )

        # Relu
        x_student_skip = self.student_proj(x_student) * 0.3
        x_course_1 = self.dropout(torch.relu(course_from_students + course_from_course))
        x_concept_1 = self.dropout(torch.relu(concepts_from_students + concepts_from_course + concepts_from_concepts))
        x_student_1 = self.layer_norm(student_from_course_l1 + student_from_concept_l1 + x_student_skip)
        x_student_1 = self.dropout(torch.relu(x_student_1))


        ## Layer 2
        student_from_course = self.conv2['rev_enrolled_in'](
            (x_course_1, x_student_1),
            data['course', 'rev_enrolled_in', 'student'].edge_index,
            # size=(x_course_1.size(0), x_student.size(0))   # ← (num_src, num_dst)
        )

        student_from_concept = self.conv2['rev_engaged_with'](
            (x_concept_1, x_student_1),
            data['concept', 'rev_engaged_with', 'student'].edge_index,
            # size=(x_concept_1.size(0), x_student.size(0))   # ← (num_src, num_dst)
        )

        x_student_1 = self.dropout(torch.relu(student_from_course + student_from_concept))

        return x_student_1, x_course_1, x_concept_1
    
model = GraphSAGE(hidden_dim=256)
out = model(data)
print(out[0].shape) 
print(out[1].shape) 
print(out[2].shape) 

torch.Size([2000, 256])
torch.Size([40, 256])
torch.Size([80, 256])


In [19]:
student_indices = torch.tensor(
    [student_id_map[s] for s in enrollments['student_id']],
    dtype = torch.long
)

course_indices = torch.tensor(
    [course_id_map[c] for c in enrollments['course_id']],
    dtype= torch.long
)

risk_map = {'Low': 0, 'Medium': 1, 'High': 2}

labels = torch.tensor(
    enrollments['risk_class'].map(risk_map).values,
    dtype=torch.long
)

print("Labels shape:", labels.shape)
print("Label distribution:")
print(enrollments['risk_class'].value_counts().sort_index())

train_mask_enrollments = train_mask[student_indices]

print(student_indices.shape, course_indices.shape, labels.shape, train_mask_enrollments.shape, train_mask_enrollments.sum())

Labels shape: torch.Size([41985])
Label distribution:
risk_class
High       3486
Low       11394
Medium    27105
Name: count, dtype: int64
torch.Size([41985]) torch.Size([41985]) torch.Size([41985]) torch.Size([41985]) tensor(29430)


In [20]:
student_embs = out[0][student_indices]

print(student_embs.shape)

torch.Size([41985, 256])


In [21]:
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)
counts = Counter(labels[train_mask_enrollments].numpy())
total = sum(counts.values())

weights = torch.tensor([
    total / (3 * counts[0]),
    total / (3 * counts[1]),
    total / (3 * counts[2]),
], dtype=torch.float)

print("Class counts:", counts)
print("Class weights:", weights)
loss_fn = torch.nn.CrossEntropyLoss(weight = weights, label_smoothing=0.2)
# loss_fn = FocalLoss(weight=weights, gamma = 1.0)

for epoch in range(251):
    model.train()
    optimizer.zero_grad()

    out = model(data)

    student_embs = out[0][student_indices]
    course_embs = out[1][course_indices]

    pairs = torch.cat([student_embs, course_embs, student_embs - course_embs, student_embs * course_embs], dim = 1)

    preds = model.task_a_head(pairs)
    loss = loss_fn(preds[train_mask_enrollments], labels[train_mask_enrollments])

    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

Class counts: Counter({np.int64(1): 18900, np.int64(0): 7926, np.int64(2): 2604})
Class weights: tensor([1.2377, 0.5190, 3.7673])
Epoch 0 | Loss: 1.2420
Epoch 50 | Loss: 0.8803
Epoch 100 | Loss: 0.8728
Epoch 150 | Loss: 0.8686
Epoch 200 | Loss: 0.8661
Epoch 250 | Loss: 0.8639


In [22]:
val_mask_enrollments = val_mask[student_indices]
test_mask_enrollments = test_mask[student_indices]

In [23]:
model.eval()
with torch.no_grad():
    out = model(data)
    student_embs = out[0][student_indices]
    course_embs = out[1][course_indices]
    pairs = torch.cat([student_embs, course_embs, student_embs - course_embs, student_embs * course_embs ], dim=1)
    preds = model.task_a_head(pairs)

    # Get predicted class
    predicted = preds.argmax(dim=1)

    # Validation set
    val_preds = predicted[val_mask_enrollments].numpy()
    val_labels = labels[val_mask_enrollments].numpy()

    print("=== Validation ===")
    print(f"Macro F1: {f1_score(val_labels, val_preds, average='macro'):.4f}")
    print(f"\n{classification_report(val_labels, val_preds, target_names=['Low', 'Medium', 'High'])}")

=== Validation ===
Macro F1: 0.7880

              precision    recall  f1-score   support

         Low       0.76      0.91      0.83      1773
      Medium       0.94      0.76      0.84      3966
        High       0.54      0.96      0.69       526

    accuracy                           0.82      6265
   macro avg       0.75      0.88      0.79      6265
weighted avg       0.86      0.82      0.83      6265



In [24]:
test_mask_enrollments = test_mask[student_indices]

model.eval()
with torch.no_grad():
    out = model(data)
    student_embs = out[0][student_indices]
    course_embs = out[1][course_indices]
    pairs = torch.cat([student_embs, course_embs, student_embs - course_embs, student_embs * course_embs ], dim=1)
    preds = model.task_a_head(pairs)
    predicted = preds.argmax(dim=1)

    test_preds = predicted[test_mask_enrollments].numpy()
    test_labels = labels[test_mask_enrollments].numpy()

    print("=== Test Set ===")
    print(f"Macro F1: {f1_score(test_labels, test_preds, average='macro'):.4f}")
    print(f"\n{classification_report(test_labels, test_preds, target_names=['Low', 'Medium', 'High'])}")

=== Test Set ===
Macro F1: 0.7738

              precision    recall  f1-score   support

         Low       0.77      0.90      0.83      1695
      Medium       0.95      0.80      0.87      4239
        High       0.46      0.94      0.62       356

    accuracy                           0.84      6290
   macro avg       0.73      0.88      0.77      6290
weighted avg       0.87      0.84      0.84      6290



In [25]:
print("=== Class-Level Accuracy ===\n")
classes = ['Low', 'Medium', 'High']
for i, class_name in enumerate(classes):
    mask = test_labels == i
    correct = (test_preds[mask] == i).sum()
    total = mask.sum()
    accuracy = correct / total
    print(f"{class_name}: {correct}/{total} correct = {accuracy:.2%}")

# Overall
print(f"\nOverall: {(test_preds == test_labels).sum()}/{len(test_labels)} = {(test_preds == test_labels).mean():.2%}")

# Confusion matrix
print("\n=== Confusion Matrix ===")
print("(rows = actual, columns = predicted)")
print(f"{'':>10} {'Low':>8} {'Medium':>8} {'High':>8}")
cm = confusion_matrix(test_labels, test_preds)
for i, class_name in enumerate(classes):
    print(f"{class_name:>10} {cm[i][0]:>8} {cm[i][1]:>8} {cm[i][2]:>8}")

=== Class-Level Accuracy ===

Low: 1531/1695 correct = 90.32%
Medium: 3396/4239 correct = 80.11%
High: 336/356 correct = 94.38%

Overall: 5263/6290 = 83.67%

=== Confusion Matrix ===
(rows = actual, columns = predicted)
                Low   Medium     High
       Low     1531      164        0
    Medium      451     3396      392
      High        0       20      336


In [26]:
# torch.save({
#     'model_state_dict': model.state_dict(),
#     'data': data,
#     'student_id_map': student_id_map,
#     'course_id_map': course_id_map,
#     'concept_id_map': concept_id_map,
# }, 'models/task_a_model.pt')

# print("Model saved!")

In [27]:
torch.save({
    'model_state_dict': model.state_dict(),
    'data': data,
    'student_id_map': student_id_map,
    'course_id_map': course_id_map,
    'concept_id_map': concept_id_map,
}, 'models/task_a_model_new.pt')

print("Model saved!")

Model saved!


In [28]:
model.eval()
with torch.no_grad():
    out = model(data)
    student_embs = out[0][student_indices]
    course_embs = out[1][course_indices]
    pairs = torch.cat([student_embs, course_embs, student_embs - course_embs, student_embs * course_embs], dim=1)
    preds = model.task_a_head(pairs)
    predicted = preds.argmax(dim=1)

    val_preds = predicted[val_mask_enrollments]

print("Prediction distribution on validation set:")
print(f"  Low: {(val_preds == 0).sum().item()}")
print(f"  Medium: {(val_preds == 1).sum().item()}")
print(f"  High: {(val_preds == 2).sum().item()}")

Prediction distribution on validation set:
  Low: 2115
  Medium: 3214
  High: 936


In [29]:
# class FocalLoss(nn.Module):
#     def __init__(self, weight = None, gamma = 2.0):
#         super().__init__()
#         self.weight = weight
#         self.gamma = gamma

#     def forward(self, preds, targets):
#         ce_loss = torch.nn.functional.cross_entropy(preds, targets, weight= self.weight, reduction = 'none')
#         pt = torch.exp(-ce_loss)
#         focal_loss = ((1 - pt) ** self.gamma) * ce_loss
#         return focal_loss.mean()